## Import Requirements

In [ ]:
import pandas as pd
import re
import numpy as np
import math

from nltk.corpus import stopwords
# nltk.download('stopwords')
import contractions
import fasttext

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

C:\Users\kylie\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Preprocessing Step

In [ ]:
# Load csv file into songs_df dataframe
songs_df = pd.read_csv("../../Tmdata/DATA/songs_with_vader.csv")

print("Songs shape:", songs_df.shape)
print("\nSongs columns:", songs_df.columns.tolist())
songs_df.head()

Songs shape: (520589, 23)

Songs columns: ['id', 'name', 'artists', 'danceability', 'energy', 'loudness', 'speechiness', 'valence', 'tempo', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'niche_genres', 'lyrics_clean', 'lyrics_normalized', 'language', 'non_english_ratio', 'word_count', 'vader_compound', 'vader_sentiment']


,id,name,artists,danceability,energy,loudness,speechiness,valence,tempo,lyrics,...,total_artist_followers,avg_artist_popularity,niche_genres,lyrics_clean,lyrics_normalized,language,non_english_ratio,word_count,vader_compound,vader_sentiment
0,0Prct5TDjAnEgIqbxcldY9,!,"[""HELLYEAH""]",0.415,0.605,-11.157,0.0575,0.193,100.059,"He said he came from Jamaica,\nhe owned a coup...",...,769490,52.0,"[""groove metal"", ""metal""]","He said he came from Jamaica,\nhe owned a coup...","he said he came from jamaica, he owned a coupl...",en,0.229474,475,0.9574,Positive
1,2ASl4wirkeYm3OWZxXKYuq,!!,"[""Yxngxr1""]",0.788,0.648,-9.135,0.3150,0.287,79.998,"Fuck the bitch, now she running with my kids\n...",...,143628,45.0,[],"Fuck the bitch, now she running with my kids\n...","fuck the bitch, now she running with my kids a...",en,0.175781,256,0.9982,Positive
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,"[""Dillinger Four""]",0.171,0.957,-5.749,0.1490,0.349,175.317,You like to stand on the other side\nPoint and...,...,36619,35.0,"[""melodic hardcore"", ""pop punk"", ""punk"", ""skat...",You like to stand on the other side\nPoint and...,you like to stand on the other side point and ...,en,0.127962,211,-0.9868,Negative
3,1xBFhv5faebv3mmwxx7DnS,!Lost!,"[""Ril\u00e8s""]",0.729,0.552,-8.562,0.0650,0.380,86.103,I would like to give you all my time\nI would ...,...,929303,63.0,"[""french rap""]",I would like to give you all my time\nI would ...,i would like to give you all my time i would l...,en,0.111732,358,-0.2434,Negative
4,0gNNToCW3qjabgTyBSjt3H,!Que Vida! - Mono Version,"[""Love""]",0.600,0.540,-11.803,0.0328,0.547,125.898,With pictures and words\nIs this communicating...,...,273598,46.0,"[""acid rock"", ""baroque pop"", ""proto-punk"", ""ps...",With pictures and words\nIs this communicating...,with pictures and words is this communicating?...,en,0.170732,123,-0.8528,Negative


In [3]:
# Inspect structure of the lyrics column
songs_df['lyrics_normalized'][0]

'he said he came from jamaica, he owned a couple acres a couple fake visas \'cause he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that\'s seldom this box better than the box he was held in i\'m momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that\'s top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i\'m in the zone they holler at me, but it\'s you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it\'s yours anywhere, every

In [4]:
# Preprocessing function
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    
    # Replace literal '\n' with actual newlines
    text = text.replace('\\n', '\n')
    
    # Split into lines
    lines = text.split('\n')
    
    cleaned_lines = []
    previous_line = None
    
    for line in lines:
        line = line.strip()
        if line != previous_line and line != '':
            # Expand contractions
            line = contractions.fix(line)
            cleaned_lines.append(line)
            previous_line = line
    
    # Join lines back with a single newline
    return '\n'.join(cleaned_lines)

# Apply to your DataFrame
songs_df['final_lyrics_cleaned'] = songs_df['lyrics_normalized'].apply(clean_lyrics)

In [5]:
# Check first song
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica, he owned a couple acres a couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that is seldom this box better than the box he was held in i am momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that is top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i am in the zone they holler at me, but it is you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it is yours anywhere, everyw

In [6]:
# Load the FastText language identification model
model = fasttext.load_model("lid.176.bin")  # make sure this path points to where you saved lid.176.bin

# Function to check if the song is English
def is_english_song(text):
    # Replace \n with space to make it one line
    text_one_line = text.replace('\n', ' ')
    lang, prob = model.predict(text_one_line)
    return lang[0] == "__label__en" and prob[0] > 0.9


print(f"Number of English songs before filtering: {len(songs_df)}")
# Apply function and filter in-place
songs_df = songs_df[songs_df['final_lyrics_cleaned'].apply(is_english_song)].reset_index(drop=True)

print(f"Number of English songs after filtering: {len(songs_df)}")

Number of English songs before filtering: 520589
Number of English songs after filtering: 453384


In [7]:
# Check result
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica, he owned a couple acres a couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that is seldom this box better than the box he was held in i am momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that is top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i am in the zone they holler at me, but it is you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it is yours anywhere, everyw

In [8]:
stop_words = set(stopwords.words('english'))

lyrics_stopwords = {
    'yeah', 'oh', 'ooh', 'ah', 'na', 'la', 'ha',
    'gonna', 'wanna', 'gotta', 'chorus', 'verse',
    'hook', 'bridge', 'repeat', 'whoa', 'aye', 'ay', 
    'owhh', 'yeahh', 'ohoh', 'ahoh', 'ohohh', 'aha', 
    'ohh', 'uh', 'huh', 'mm', 'mmm', 'woo', 'yep', 'yup', 
    'hmm', 'hmmm', 'uhh', 'uhhh', 'huhh','ohoh', 'woooooa', 
    'da', 'ba', 'doo', 'blah', 'dum', 'ya', 'de', 'du', 'dee', 'bum',
    'ca', 'wo', 'wan', 'gon', 'ta', 'ai', 'uh', 'ya', 'yo', 'ai-ai',
    'la-la-la', 'na-na-na', 'la-la', 'na-na', 'oooooooooh', 'ooooooooooooooh',
    'mmmmmmmm', 'boo-boo-boo-boo-yah', 'oh-oh-oh-ohh'
}

all_stopwords = stop_words.union(lyrics_stopwords)

def clean_lyrics_keep_lines(text):
    lines = text.split('\n')  # keep each line
    cleaned_lines = []
    
    for line in lines:
        line = line.lower().strip()
        # remove punctuation/numbers, keep spaces
        line = re.sub(r'[^a-z\s]', ' ', line)
        # split into tokens, remove stopwords and single letters
        tokens = [t for t in line.split() if t and t not in all_stopwords and len(t) > 1]
        if tokens:  # only add non-empty lines
            cleaned_lines.append(' '.join(tokens))

    # final check to remove any leftover empty lines
    cleaned_lines = [line for line in cleaned_lines if line.strip()]
    return '\n'.join(cleaned_lines)

songs_df['final_lyrics_cleaned'] = songs_df['final_lyrics_cleaned'].apply(clean_lyrics_keep_lines)

In [9]:
# Check result
print(songs_df['final_lyrics_cleaned'].iloc[0])

said came jamaica owned couple acres couple fake visas never got papers gave love fucking heart breakers getting money movers shakers mixed couple things bald like couple rings bricks condo grams sing sing left arm baby mother tatted year bid north ratted anyway felt helped put lock seat belt took belgium welcome bitches pretty seldom box better box held momma order call daddy like daughters like get drunk like sober top toppa never fuck beginners let play pussy lick fingers zone holler high school crew slide give whenever want whip whenever want baby anywhere everywhere baby world alright baby world got nigga home one side best friend dyke fucked around times momma alike fight tell make money tell make wife tell bitch crazy fuck wrong excuse french long kisser try tell one hitting say niggas say niggas right tonight put something tight judge would get life love like brother fuck like husband pussy like oven hot put tongue rub genie bottle pussy wet need goggles tell mine tell stop lyi

#### Applying the Model

In [10]:
documents = songs_df['final_lyrics_cleaned'].tolist()
print(f"Number of documents: {len(documents)}")
print(f"Sample document length: {len(documents[0])} characters")

Number of documents: 453384
Sample document length: 1230 characters


In [11]:
sample_df = songs_df.sample(20000, random_state=42).reset_index(drop=True)
sample_documents = sample_df['final_lyrics_cleaned'].tolist()

print(f"Number of sampled documents: {len(sample_documents)}")
print(f"Sample document length: {len(sample_documents[0])} characters")

Number of sampled documents: 20000
Sample document length: 750 characters


In [12]:
def chunk_lyrics(documents, min_words=12):
    """
    Splits a list of cleaned song strings into fixed-size word chunks.

    Parameters
    ----------
    documents : list of str   — one cleaned song per element
    min_words : int           — minimum words per chunk

    Returns
    -------
    chunks   : list of str   — all text chunks
    song_map : list of int   — index into `documents` for each chunk

    Works for both training (full list) and demo (single song):
        training : chunk_lyrics(sample_documents)
        demo     : chunk_lyrics([preprocess_single_song(raw_lyrics)])
    """
    chunks = []
    song_map = []
    for idx, song in enumerate(documents):
        lines = song.split('\n')
        buffer = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            buffer.extend(line.split())
            while len(buffer) >= min_words:
                chunks.append(' '.join(buffer[:min_words]))
                song_map.append(idx)
                buffer = buffer[min_words:]
        if buffer:
            if chunks and song_map[-1] == idx:
                chunks[-1] += ' ' + ' '.join(buffer)
            else:
                chunks.append(' '.join(buffer))
                song_map.append(idx)
    return chunks, song_map


sample_line_documents, song_map = chunk_lyrics(sample_documents, min_words=12)

print(f"Total chunks: {len(sample_line_documents)}")
print(f"Sample chunk: {sample_line_documents[0]}")

Total chunks: 183317
Sample chunk: little sleepy boy know time well hour bedtime long past though know


In [13]:
# Inspect chunks for the first song
first_song_index = 0
first_song_chunks = [
    chunk for chunk, idx in zip(sample_line_documents, song_map)
    if idx == first_song_index
]
for i, chunk in enumerate(first_song_chunks, start=1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: little sleepy boy know time well hour bedtime long past though know
Chunk 2: fighting tell rub eyes fading fast fading fast run come see st
Chunk 3: judy comet roll across skies leave spray diamonds wake long see st
Chunk 4: judy comet sparkle eyes awake little boy lay body little boy close
Chunk 5: weary eyes nothing flashing fireflies well sang sang twice going sing three
Chunk 6: times going stay til resistance overcome cannot sing boy sleep well makes
Chunk 7: famous daddy look dumb run come see st judy comet roll across
Chunk 8: skies leave spray diamonds wake long see st judy comet sparkle eyes
Chunk 9: awake little boy little boy lay body little boy little boy close
Chunk 10: weary eyes nothing flashing fireflies little sleepy boy know time well hour bedtime long past though know fighting tell rub eyes fading fast


In [14]:
# 1. Initialise embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Compute embeddings in batches
batch_size = 256
all_embeddings = []
num_batches = math.ceil(len(sample_line_documents) / batch_size)

for i in range(num_batches):
    batch_lines = sample_line_documents[i*batch_size : (i+1)*batch_size]
    batch_embeddings = embedding_model.encode(batch_lines, show_progress_bar=True)
    all_embeddings.append(batch_embeddings)

all_embeddings = np.vstack([b.astype(np.float32) for b in all_embeddings])
print(f"Shape of embeddings: {all_embeddings.shape}")

# 3. Initialise BERTopic with UMAP
umap_model = UMAP(n_neighbors=80, n_components=8, low_memory=True)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    min_topic_size=100,
    top_n_words=10,
    nr_topics=None,
    calculate_probabilities=True,
    verbose=True
)

# 4. Fit BERTopic
topics, probs = topic_model.fit_transform(
    sample_line_documents,
    embeddings=all_embeddings
)

print("\nBefore outlier reduction:")
print(f"Number of topics: {len(set(topics)) - (1 if -1 in set(topics) else 0)}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.09it/s]
2026-04-13 21:27:25,714 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Shape of embeddings: (183317, 384)


2026-04-13 21:30:42,062 - BERTopic - Dimensionality - Completed ✓
2026-04-13 21:30:42,070 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 21:32:54,893 - BERTopic - Cluster - Completed ✓
2026-04-13 21:32:54,942 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-13 21:32:56,518 - BERTopic - Representation - Completed ✓



Before outlier reduction:
Number of topics: 66


After training, we have:
- `topic_model` → the trained BERTopic model
- `topics` → topic ID for each chunk (before outlier reduction)
- `probs` → confidence scores
- `all_embeddings` → vectors (expensive to recompute — keep these!)
- `sample_line_documents` → actual text chunks
- `song_map` → mapping chunk index → song index in sample_df

In [15]:
topics_array = np.array(topics)
print(f"Outliers before reduction: {(topics_array == -1).sum()}")

# 5. Reduce outliers
topics_reduced = topic_model.reduce_outliers(
    sample_line_documents,
    topics,
    strategy="c-tf-idf"
)

print("\nAfter outlier reduction:")
print(f"Number of topics: {len(set(topics_reduced)) - (1 if -1 in set(topics_reduced) else 0)}")
print(f"Outliers remaining: {(np.array(topics_reduced) == -1).sum()}")

# 6. Update topic assignments in model
topic_model.update_topics(sample_line_documents, topics=topics_reduced)

# 7. Final distribution
print("\nUpdated topic distribution (top 10):")
print(pd.Series(topics_reduced).value_counts().sort_index().head(10))

Outliers before reduction: 138909


2026-04-13 21:32:59,823 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



After outlier reduction:
Number of topics: 66
Outliers remaining: 15

Updated topic distribution (top 10):
-1       15
 0    22253
 1    12102
 2     7667
 3     3984
 4     3275
 5     3213
 6     2951
 7     3425
 8    20188
Name: count, dtype: int64


In [ ]:
line_df_v3 = pd.DataFrame({
    "song_idx": song_map,
    "topic": topics_reduced
})

In [17]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(20))

topic_id = 0
print(topic_model.get_topic(topic_id))

    Topic  Count                              Name  \
0      -1     15  -1_ravioli_pocketioli_poodle_yuk   
1       0  22253            0_love_baby_want_heart   
2       1  12102         1_nigga_money_niggas_shit   
3       2   7667            2_god_lord_jesus_faith   
4       3   3984           3_sea_ocean_river_water   
5       4   3275           4_drink_wine_drunk_beer   
6       5   3213   5_dream_dreams_dreaming_dreamed   
7       6   2951         6_hair_shoes_wear_clothes   
8       7   3425          7_mama_mother_father_son   
9       8  20188              8_time_think_go_life   
10      9   3234         9_fire_burn_burning_flame   
11     10   4665          10_ride_road_train_drive   
12     11   4500         11_believe_lies_truth_lie   
13     12   1803      12_dance_dancing_night_floor   
14     13   1155     13_christmas_santa_year_merry   
15     14   2158            14_moon_stars_star_sky   
16     15   2765           15_rock_roll_beat_music   
17     16   4025    16_light

In [19]:
topic_id = -1
print(topic_model.get_topic(topic_id))

[('ravioli', 1.8920077471629535), ('pocketioli', 0.871478759601812), ('poodle', 0.724290461833334), ('yuk', 0.45385612771384554), ('ites', 0.4171033383348772), ('ska', 0.40559098446045144), ('pocketioliravioli', 0.38474799997567727), ('chews', 0.36415116321013985), ('dja', 0.33763214933555613), ('guffaw', 0.26910109399843074)]


In [ ]:
for i in range(0, 17):
    song = line_df_v3[line_df_v3['song_idx'] == i]
    if song.empty:
        print(f"Song {i}: No data\n")
        continue
    topic_counts = song['topic'].value_counts()
    dominant_topic = topic_counts.idxmax()
    print(f"Song {i}")
    print(topic_counts)
    print(f"Dominant topic: {dominant_topic}\n")

Song 0
topic
55    3
21    3
34    2
8     1
57    1
Name: count, dtype: int64
Dominant topic: 55

Song 1
topic
3     2
43    2
10    1
55    1
33    1
Name: count, dtype: int64
Dominant topic: 3

Song 2
topic
0     7
5     3
20    3
13    2
61    1
1     1
38    1
21    1
Name: count, dtype: int64
Dominant topic: 0

Song 3
topic
62    4
9     2
Name: count, dtype: int64
Dominant topic: 62

Song 4
topic
0    4
Name: count, dtype: int64
Dominant topic: 0

Song 5
topic
0     9
43    2
12    1
62    1
19    1
8     1
Name: count, dtype: int64
Dominant topic: 0

Song 6
topic
2     1
0     1
10    1
30    1
46    1
Name: count, dtype: int64
Dominant topic: 2

Song 7
topic
8     5
0     3
11    1
40    1
47    1
Name: count, dtype: int64
Dominant topic: 8

Song 8
topic
43    6
0     1
Name: count, dtype: int64
Dominant topic: 43

Song 9
topic
1     13
4      2
0      2
6      1
21     1
46     1
3      1
38     1
41     1
26     1
42     1
8      1
25     1
54     1
Name: count, dtype: int64

In [ ]:
# Flag songs where no topic appears more than once (potentially noisy)
song_chunks_grouped = line_df_v3.groupby('song_idx')
noisy_songs = song_chunks_grouped.filter(lambda x: x['topic'].value_counts().max() == 1)
print(f"Songs with no dominant topic: {noisy_songs['song_idx'].nunique()}")

Songs with no dominant topic: 2557


In [25]:
for i in range(len(topic_model.get_topics())):
    print(topic_model.get_topic(i))

[('love', 0.041203306160697194), ('baby', 0.03565130848907291), ('want', 0.02124145478373855), ('heart', 0.02005324911045949), ('know', 0.01895815047482205), ('girl', 0.016983932420462425), ('need', 0.01689571221433005), ('give', 0.014709259549371068), ('say', 0.013697617468597977), ('feel', 0.012903029638656187)]
[('nigga', 0.04526768390256905), ('money', 0.03578090566463457), ('niggas', 0.035157168431549604), ('shit', 0.0322334419859999), ('fuck', 0.028554905484202857), ('bitch', 0.028542911758753814), ('get', 0.018257118684863304), ('got', 0.01730613450161281), ('like', 0.014405068763080911), ('ass', 0.01342283121754721)]
[('god', 0.055582426502636666), ('lord', 0.036270519941404296), ('jesus', 0.03344671976697769), ('faith', 0.021317198781411983), ('sing', 0.017961989508274823), ('pray', 0.017361504702857456), ('praise', 0.016853164754623014), ('holy', 0.016725956594256252), ('king', 0.016319311308431023), ('name', 0.01617535615145197)]
[('sea', 0.06194687419889168), ('ocean', 0.03

In [26]:
print(f"Topics distribution:\n{pd.Series(np.array(topics_reduced)).value_counts().sort_values(ascending=False).head(100)}")

Topics distribution:
 0     22253
 8     20188
 1     12102
 2      7667
 21     5211
       ...  
 54      543
 50      369
 65      315
 60      308
-1        15
Name: count, Length: 67, dtype: int64


**Observation:** Embeddings are too scattered → assigns outliers.
Possible root cause: chunks are still not semantically strong enough / too different from one another → embeddings end up spread out.

## Evaluation Metrics

In [32]:
# ── Coherence Score ──────────────────────────────────────────────────────────
tokenized_docs = [doc.split() for doc in sample_line_documents]
dictionary = Dictionary(tokenized_docs)

topics_dict = topic_model.get_topics()
topic_words = []
topic_ids_eval = []

for topic_id, words_weights in topics_dict.items():
    if topic_id != -1 and len(words_weights) >= 10:
        words = [word for word, _ in words_weights[:15]]
        topic_words.append(words)
        topic_ids_eval.append(topic_id)

coherence_model_cv = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
overall_coherence = coherence_model_cv.get_coherence()
print(f"Overall Coherence Score: {overall_coherence:.4f}")


# ── Topic Diversity ───────────────────────────────────────────────────────────
def compute_topic_diversity(topic_model, top_k=15):
    topics = topic_model.get_topics()
    all_words = []
    for topic_id, words in topics.items():
        if topic_id == -1:
            continue
        all_words.extend([word for word, _ in words[:top_k]])
    return len(set(all_words)) / len(all_words)

diversity_score = compute_topic_diversity(topic_model)
print(f"Topic Diversity Score: {diversity_score:.4f}")


Overall Coherence Score: 0.4616
Topic Diversity Score: 0.8091
